In [ ]:
# Starting the omlx server
!/opt/homebrew/bin/omlx start

In [ ]:
#import statements and keys

import os
from pathlib import Path
from dotenv import load_dotenv
import os
import yaml
import logging
import re
from datetime import datetime
import ast
import yaml
import time
from openai import OpenAI
import ast
import requests
import json
from groq import Groq

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
env_path = Path.cwd().parent / "environment.env"

# Load the .env file
if load_dotenv(dotenv_path=env_path):
    print("Environment variables loaded successfully!")
else:
    print("Error: Could not find or load the .env file.")

OmlxAPI = os.getenv('OmlxAPI')
baseurl = os.getenv('baseurl')
API = os.getenv('API')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')



Environment variables loaded successfully!


In [ ]:
# Local LLM

def llm_response_local(prompt: str, ) -> str:
    """Generic function to get a response from the Groq Qwen LLM."""
    try:
        client = OpenAI(
            base_url=baseurl,  # Your oMLX local address
            api_key=OmlxAPI          # Your oMLX API Key
        )
        response = client.chat.completions.create(
            model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
            messages=[{"role": "user", "content": prompt}]
        )
             
        content = response.choices[0].message.content
        return content
    except Exception as e:
        logging.error(f"An error occurred while communicating with the Omlx API: {e}")
        return ""




In [ ]:
#llm response from open router

def LlmResponseOpenRouter(prompt):
    try:
        # First API call with reasoning
        response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": "google/gemma-4-26b-a4b-it",
            "messages": [
                {
                "role": "user",
                "content": f"{prompt}"
                }
            ],
            "reasoning": {"enabled": True}
        })
        )

        # Extract the assistant message with reasoning_details
        response = response.json()
        response = response['choices'][0]['message']['content']
        return response
    except Exception as e:
            logging.error(f"An error occurred while communicating with the Openrouter API: {e}")
            return ""

In [ ]:
#Response from groq

def llmresponsegroq(prompt):
    try:
        client = Groq(api_key=GROQ_API_KEY)
        completion = client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6,
            max_completion_tokens=2048,
            top_p=0.95,
            reasoning_effort="default",
            stream=True,
            # stream_options={"include_usage": True} # Ensures usage metadata is sent in the stream
        )
        return completion
    except Exception as e:
          logging.error(f"An error occurred while communicating with the Omlx API: {e}")
          return ""


In [30]:
completion = llmresponsegroq("hi, tell me a joke")

for chunk in completion:
    # 1. Print content if it exists
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
        
    # 2. Check for usage metrics in the final chunk
    if hasattr(chunk, 'x_groq') and chunk.x_groq and chunk.x_groq.usage:
        print("\n\n--- Stream Usage Stats ---")
        print(f"Prompt Tokens: {chunk.x_groq.usage.prompt_tokens}")
        print(f"Completion Tokens: {chunk.x_groq.usage.completion_tokens}")
        print(f"Total Tokens Used: {chunk.x_groq.usage.total_tokens}")


2026-09-06 20:50:15,956 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Why did the scarecrow win an award?

Because he was outstanding in his field! 🌾

--- Stream Usage Stats ---
Prompt Tokens: 18
Completion Tokens: 22
Total Tokens Used: 40


In [42]:
groqlimits = []
groqlimits.append(chunk.x_groq.usage.total_tokens)

In [45]:
groqlimits

[40]

In [ ]:
response = LlmResponseOpenRouter("tell me a joke")
try:
    experience = ast.literal_eval(response)
except:
    experience = response

In [ ]:
# def yamlcheck(content):
#     try:
#         raw_response = content.strip()
#     except:
#         None

#     # Remove markdown code fences if Qwen accidentally includes them
#     if raw_response.startswith("```"):
#         raw_response = (raw_response.replace("```python", "").replace("```", "").strip()
#         )

#     try:
#         # Safely convert the string representation of a list into a real Python list
#         content = ast.literal_eval(raw_response)
#     except Exception as e:
#         print(f"Error parsing list: {e}")
#         # Fallback: just split by lines if it completely fails
#         content = [line.strip() for line in raw_response.split("\n") if line.strip()]
        
#     return content

In [51]:
path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)

In [ ]:
jobdescription = input('Enter the job description')

In [ ]:
#First Prompt

path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
with open(path, 'r') as f:
    resume = yaml.safe_load(f)

path = '/Users/karthik/Documents/Github/Colab/output.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
    
prompt = f""""

rewrite the points to suit the job description and resume and sort based on importance to the job and select top 7-9 per title in a dictionary

[Constraints]
- Truthfulness: Use ONLY metrics, tools, and software explicitly in the Resume. Never add/change tools or numbers.
- Format: No markdown blocks (```), no \n, no indents.
- No Citation
- Content: Blend problem, solution, and impact seamlessly. No labels like "Problem:" or "Impact:".

NO PREAMBLE

[Data]
Resume: {resume}
JobDescription: {jobdescription}        

[Guardrail] Ensure exactly suited points per title. Zero invented skills/tools.

"""

In [ ]:


# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {API}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "google/gemma-4-26b-a4b-it",
    "messages": [
        {
          "role": "user",
          "content": f"{prompt}"
        }
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']['content']
experience = ast.literal_eval(response)


2026-09-06 18:50:42,286 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In silence, keys strike glass and light,
A digital dawn breaks through the night.
Logic flows like water, clear and deep,
While bugs hide in the shadows, fast asleep.

One missing bracket, a semicolon’s sigh,
And chaos blooms beneath a clear blue sky.
But when the error clears, the screen turns green,
The cleanest code I’ve ever seen.

--- Stream Usage Stats ---
Prompt Tokens: 19
Completion Tokens: 83
Total Tokens Used: 102


In [7]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
completion = client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=[
      {
        "role": "user",
        "content": ""
      }
    ],
    temperature=0.6,
    max_completion_tokens=2048,
    top_p=0.95,
    reasoning_effort="default",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


2026-09-06 18:50:24,126 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi"
   - This is a simple greeting.

2.  **Identify Intent:**
   - The user is initiating a conversation.
   - No specific question or task is provided.

3.  **Determine Response Strategy:**
   - Acknowledge the greeting warmly.
   - Offer assistance.
   - Keep it open-ended to encourage the user to share what they need.

4.  **Draft Response (Mental Refinement):**
   - "Hello! How can I assist you today?"
   - "Hi there! What can I help you with?"
   - "Hey! I'm here to help. What's on your mind?"

   All are good. I'll go with a friendly, professional, and open-ended response.

5.  **Final Output Generation:**
   - "Hello! How can I assist you today?" (Simple, clear, and inviting)✅
   - Matches standard AI greeting protocols.
   - Ready.✅
</think>

Hello! How can I assist you today? 😊

In [ ]:
# experience.keys()
if (len(experience.keys()) ==len(test['work_experience'])):
    for pos, (key, value) in enumerate(experience.items()):
        # print(f"Position: {pos}")
        # print(f"Value: {value}")
        test['work_experience'][pos]['achievements'] = value

    # Now saving to YAML will be perfectly clean and structured
    with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

    print("inital tailored resume collected")
else:
    print("There is not same number of experiences in source and destination")

    

In [ ]:
# #Load the content files

# path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
# with open(path, 'r') as f:
#     resume = yaml.safe_load(f)
# # a = list(resume['Professional Experience'].keys())

# #Load Destination Files
# path = '/Users/karthik/Documents/Github/Colab/output.yaml'
# with open(path, 'r') as f:
#     test = yaml.safe_load(f)



# #Create loop to get tailored initial responses with checking all the experiences are available

# if (len(resume["work_experience"]) ==len(test['work_experience'])):
#     for i in range(len(resume["work_experience"])):
#     # for i in range(1):
#         experience = resume["work_experience"][i]["achievements"]
#         prompt1 = f"""[Task] rewrite the points to suit the job description and resume and sort based on importance to the job and select top 7 

#         [Constraints]
#         - Truthfulness: Use ONLY metrics, tools, and software explicitly in the Resume. Never add/change tools or numbers.
#         - Format: Return ONLY a raw, valid Python list of 7 strings on ONE continuous line. No markdown blocks (```), no \n, no indents. Start with [ and end with ].
#         - Content: Blend problem, solution, and impact seamlessly. No labels like "Problem:" or "Impact:".
        
#         NO PREAMBLE

#         [Data]
#         Resume: {experience}
#         Job: {jobdescription}

#         [Guardrail] Ensure exactly 7 elements are in the list. Zero invented skills/tools."""

    
#         tailoredresponse = llm_response_qwen(prompt1)
#         cleanedresponse = yamlcheck(tailoredresponse)

#         test['work_experience'][i]['achievements'] = cleanedresponse

#     # Now saving to YAML will be perfectly clean and structured
#     with open("output.yaml", "w", encoding="utf-8") as f:
#         yaml.dump(test, f, sort_keys=False, default_flow_style=False)

#     print("inital tailored resume collected")
# else:
#     print("There is not same number of experiences in source and destination")

In [72]:
path = '/Users/karthik/Documents/Github/Colab/output.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)

output = [{"name": "Core Competencies", "keywords": [ "Strategic Procurement", "Supplier Negotiation", "Procurement Analytics"]},{"name": "Core Competencies", "keywords": [ "Strategic Procurement", "Supplier Negotiation", "Procurement Analytics"]}]


skillsprompt = f""" Rewrite the skills section in a crisp one or two word to suit the below tailored resume and job description maintain same yaml format can modify or create new skill sections and rate the final skills plus reume for ats compatablity to the job description.
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

skills: {test['skills']}
Job: {jobdescription}
resume: {test['work_experience']}

Example:
OUTPUT : maintain same format

NO PREAMBLE
"""


tailoredresponse = llm_response_qwen(skillsprompt)
cleanedresponse = yamlcheck(tailoredresponse)
test['skills'] = cleanedresponse[0]['skills']
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-08-03 19:17:17,359 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [73]:
path = '/Users/karthik/Documents/Github/Colab/output.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
output = {"summary":"Strategic Procurement and Sourcing Professional with over 7 years of experience in "}

summaryprompt = f"""rewrite the summary section to suit the job description and resume
- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

summary: {test['summary']}
Job: {jobdescription}
resume: {test['work_experience']}



NO PREAMBLE
"""

tailoredresponse = llm_response_qwen(summaryprompt)
cleanedresponse = yamlcheck(tailoredresponse)
try:
    cleanedresponse[0]['summary']
except:
      test['summary'] = cleanedresponse['summary']
      
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)


2026-08-03 19:17:39,482 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
# cleanedresponse

In [ ]:
# cleanedresponse = yamlcheck(tailoredresponse)
# try:
#     cleanedresponse[0]['summary']
# except:
#       test['summary'] = cleanedresponse['summary']
      
# with open("output.yaml", "w", encoding="utf-8") as f:
#         yaml.dump(test, f, sort_keys=False, default_flow_style=False)

In [74]:
path = '/Users/karthik/Documents/Github/Colab/output.yaml'
with open(path, 'r') as f:
    test = yaml.safe_load(f)
    
output = {"project_name": "Procurement Analytics Dashboard",
    "description": "Developed a Power BI dashboard to track procurement KPIs, supplier performance, and category spend, providing actionable insights for informed purchasing decisions.",
    "keywords" : ["Power BI", "Procurement", "Supply Chain", "KPIs"]}

projectprompt = f"""Select/ Rewrite 4 projects from the below list to suit the job description

- Format: Return ONLY a raw, valid YAML format. No markdown blocks (```), no \n, no indents. Start with [ and end with ].

Output : {output}

projects: {resume['projects']}
Job: {jobdescription}

NO PREAMBLE

"""

tailoredresponse = llm_response_qwen(projectprompt)
cleanedresponse = yamlcheck(tailoredresponse)

test['projects']= cleanedresponse
with open("output.yaml", "w", encoding="utf-8") as f:
        yaml.dump(test, f, sort_keys=False, default_flow_style=False)

2026-08-03 19:17:52,711 - INFO - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"


In [75]:
!/opt/homebrew/bin/omlx restart


oMLX server restarted on port 8000


In [3]:
#make the pdf
!python ReportLabs.py 

✅ Resume generated: Karthikeyan_Baskaran_Resume.pdf


In [ ]:
# import datetime

# # Get current date
# now = datetime.datetime.now()

# # Add 2 months and handle calendar wrap-around
# future_month_num = (now.month + 2 - 1) % 12 + 1
# future_date = now.replace(month=future_month_num)

# # Print the name
# print(future_date.strftime("%b 2026"))



Oct 2026


In [ ]:
# test = {"Procurement Buyer": ["Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk.", "Led procurement reporting automation using SQL and Python, improving spend visibility and reducing manual reporting effort by 30% to support strategic sourcing decisions.", "Built custom SQL-based tracking systems that automated supplier performance monitoring, cutting manual reporting time by 50% and providing early warnings for dependency risks.", "Analyzed procurement transactions to identify spend anomalies and supplier pricing inconsistencies, improving financial controls and reducing investigation time by 50%.", "Developed procurement demand forecasting models that improved purchasing decisions, reduced excess inventory by 18%, and strengthened supplier planning.", "Boosted vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries.", "Led cross-functional 'Continuous Improvement' sessions, collaborating with stakeholders to bridge gaps between departments and implement procurement tools that scaled with the business."], "Procurement Engineer": ["Applied market intelligence and value analytics to category planning, generating insights that drove innovative sourcing strategies and achieved 12% cost savings.", "Achieved a 15% reduction in procurement costs by developing a data-driven supplier evaluation and conducting detailed cost analysis in supply chain operations.", "Mitigated supply chain risks by analyzing data to identify alternative sourcing options, addressing potential bottlenecks in procurement and delivery.", "Negotiated supplier contracts to achieve significant cost savings while ensuring compliance with procurement policies and regulations.", "Delivered savings of $35M by developing Python-based automation tools to optimize procurement and buying workflows, improving operational efficiency.", "Enhanced supplier performance monitoring by implementing vendor scorecards using Excel and Power BI, providing actionable insights for procurement decisions.", "Streamlined RFQ processes using advanced quote comparison tools and automation, reducing purchasing cycle times by 40%."], "Procurement Specialist": ["Led RFx processes for supply chain segments, negotiating competitive pricing with global vendors and achieving an 8% reduction in procurement costs.", "Identified savings of 3M EUR by analyzing supplier costs and optimizing payment credit periods, implementing efficient procurement strategies.", "Collaborated with project managers to plan budgets, coordinate bid processes, and conduct bid evaluations, ensuring cost-effective procurement aligned with project goals.", "Developed dashboards using Python and Power BI to monitor procurement KPIs, enhancing decision-making accuracy in purchasing by 18%.", "Reduced procurement costs by 10% through data-driven process optimization in supply chain and purchasing operations.", "Led process governance initiatives by standardizing procurement templates and tools, improving compliance and accelerating cross-functional alignment by 20%.", "Integrated ERP tools (SAP) to streamline procurement and supply chain workflows, reducing manual effort by 30% and improving data accuracy."], "Procurement Analyst": ["Managed RFx processes across supply chain segments, achieving 10–15% cost reductions through competitive vendor sourcing and strategic procurement.", "Performed zero-based costing using advanced Excel (macros, pivot tables) and analytical tools, enabling 15% cost savings in procurement through data-driven negotiations.", "Engaged in third-party risk management by conducting supplier risk assessments and integrating findings into category strategies, mitigating compliance and supply continuity risks.", "Built a robust supplier base by inducting vendors, implementing double sourcing, and driving localization, improving supply chain efficiency by 20%.", "Performed cost benchmarking for supply chain components, using market data to identify outliers and inform pricing strategies in purchasing.", "Executed supplier auctions using digital platforms, reducing procurement costs by 8–12% through competitive bidding and strategic negotiations.", "Built interactive dashboards to monitor procurement KPIs and supplier performance, reducing reporting time in purchasing by 40%."]}

In [ ]:
# for i in test.keys():
#     for j in range(len(test[i])):
#         print(test[i][j])
#         # print(j)

In [ ]:
# test["Procurement Buyer"][0]

In [ ]:
jobdescription


In [ ]:
# import yaml
# path = '/Users/karthik/Documents/Github/Colab/output copy.yaml'
# with open(path, 'r') as f:
#     test = yaml.safe_load(f)

In [ ]:
# prompt = f""" match between the resume and the job give me a score out of 10
#     Resume: {test}
#     Job: {jobdescription}
# """

In [ ]:
# prompt

<!-- prompt -->

In [ ]:
# from openai import OpenAI

# client = OpenAI(
#     base_url="http://127.0.0.1:8000/v1",  # Your oMLX local address
#     api_key="1234"          # Your oMLX API Key
# )

# response = client.chat.completions.create(
#     model="mlx-community/gemma-4-12B-it-OptiQ-4bit",           # Replace with your loaded model ID
#     messages=[{"role": "user", "content": prompt}]
# )

# print(response.choices[0].message.content)

In [ ]:
# type(response)

In [ ]:
# # 1. Extract the raw string from the object
# raw_content = response.choices[0].message.content

# print(raw_content)

In [ ]:
# raw_content

In [ ]:
# content = ""
# for chunk in response:
#     print(chunk)
#     break

#     # c = chunk.choices[0].delta.content if chunk.choices[0].delta.content else ""
#     # content += c

In [ ]:
# path = '/Users/karthik/Documents/Github/Colab/Resume.yaml'
# with open(path, 'r') as f:
#     resume = yaml.safe_load(f)

In [ ]:
# resume

In [ ]:
# answer = ["Automated supplier performance monitoring and provided early warnings for dependency risks by building custom SQL-based tracking systems, cutting manual reporting time by 50%.", "Developed procurement and contract documents by designing interactive Power BI dashboards that transformed messy procurement data into actionable insights to speed up sourcing workflows.", "Maintained a 'single source of truth' for end-to-end supply chain visibility and data accuracy within Sage ERP to support leadership decision-making.", "Streamlined data mining and reporting processes by engineering complex queries to transform raw numbers into monthly performance reports.", "Reduced stockouts and achieved 98% production uptime by re-engineering purchase planning for global suppliers with long lead times.", "Spearheaded strategic sourcing initiatives that slashed procurement costs while diversifying the supplier base to lower organizational risk.", "Improved vendor accountability and lead-time reliability through a high-engagement relationship model, resulting in a 30% improvement in on-time deliveries."]

In [ ]:
# answer